# Large Language Model Application  
## 以 Hugging Face Transformers Pipeline API 為例：Zero-shot Classification 零樣本分類

本教材介紹 Hugging Face `transformers` 函式庫中的 `pipeline` API，並聚焦於 **Zero-shot Classification（零樣本分類）** 的基本概念與常用參數。

Zero-shot Classification 是一種不需要針對特定分類任務重新訓練模型的文字分類方法。使用者只需要提供：

1. 一段欲分類的文字
2. 一組候選標籤 `candidate_labels`

模型便會根據文字語意，判斷該文字最可能屬於哪一個標籤。

在實務應用中，這種方法適合用於快速建立分類原型，例如：

- 新聞標題分類
- 客戶意見分類
- 商品描述分類
- 文件主題判斷
- 使用者問題意圖辨識

本範例將使用公開、客觀且去識別化的新聞標題作為示範資料。

In [2]:
# 從 transformers 函式庫匯入 pipeline
from transformers import pipeline

## 建立 Zero-shot Classification Pipeline

我們建立一個零樣本分類器。

本教材使用 `facebook/bart-large-mnli` 作為示範模型。  
這是一個常見的 NLI-based zero-shot classification 模型，適合用來展示文字與候選標籤之間的語意匹配。

第一次執行時，程式會從 Hugging Face Hub 下載模型，因此可能需要一些時間。

In [3]:
# 建立 zero-shot classification pipeline
classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli"
)

c:\Users\mapd\OneDrive - ispan.com.tw\桌面\2_教學研發\05_Natural Language Processing\3_Code4Test\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mapd\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 515/515 [00:02

## 準備欲分類的文字資料

本範例使用一則公開、客觀且去識別化的新聞標題作為輸入文字。

範例文字：

> Global chip manufacturers increase investment in advanced semiconductor production.

此句描述全球晶片製造商增加先進半導體生產投資。  
接下來，我們會讓模型判斷這段文字最接近哪一個主題分類。

In [8]:
# 設定欲分類的文字，使用客觀且去識別化的新聞標題
text = "Global chip manufacturers increase investment in advanced semiconductor production."

##  設定候選標籤 candidate_labels

`candidate_labels` 是 Zero-shot Classification 中最重要的參數之一。

它代表「模型可以選擇的分類標籤」。  
與傳統監督式分類不同，這些標籤不一定需要出現在模型訓練資料中。

例如，我們可以設定以下候選標籤：

- technology
- sports
- finance
- healthcare
- education

模型會逐一評估文字與每個標籤之間的語意關聯，並輸出每個標籤的信心分數。

In [9]:
# 設定候選分類標籤
candidate_labels = [
    "technology",
    "sports",
    "finance",
    "healthcare",
    "education"
]

## 執行零樣本分類

接下來，將文字與候選標籤傳入分類器。

`classifier()` 的基本參數如下：

- `sequences`：欲分類的文字，可以是單一字串或多筆文字
- `candidate_labels`：候選標籤清單
- `multi_label`：是否允許多個標籤同時成立

在本範例中，先使用預設設定。  
預設情況下，`multi_label=False`，模型會將候選標籤視為彼此競爭的類別，也就是輸出分數會接近一組機率分布。

In [10]:
# 執行 zero-shot classification
result = classifier(
    sequences=text,
    candidate_labels=candidate_labels
)

# 顯示原始輸出結果
result

{'sequence': 'Global chip manufacturers increase investment in advanced semiconductor production.',
 'labels': ['technology', 'finance', 'sports', 'education', 'healthcare'],
 'scores': [0.9747821092605591,
  0.01063802745193243,
  0.0059220874682068825,
  0.004432364366948605,
  0.004225434735417366]}

## 理解輸出結果

Zero-shot Classification 的輸出通常包含三個主要欄位：

- `sequence`：原始輸入文字
- `labels`：依照信心分數由高到低排序後的標籤
- `scores`：每個標籤對應的信心分數

其中，`labels[0]` 通常代表模型認為最可能的分類結果。  
`scores[0]` 則是該分類結果的信心分數。

接下來，我們將結果格式化。

In [11]:
# 取出排序後的標籤與分數
labels = result["labels"]
scores = result["scores"]

# 格式化輸出分類結果
print("輸入文字：")
print(result["sequence"])
print()

print("分類結果：")
for label, score in zip(labels, scores):
    # 將信心分數轉換為百分比格式
    print(f"{label:12s} : {score:.2%}")

輸入文字：
Global chip manufacturers increase investment in advanced semiconductor production.

分類結果：
technology   : 97.48%
finance      : 1.06%
sports       : 0.59%
education    : 0.44%
healthcare   : 0.42%


##  解讀預測結果

若模型輸出結果中，`technology` 的分數最高，代表模型認為該新聞標題最接近「科技」主題。

這是合理的，因為輸入文字中包含：

- chip manufacturers
- investment
- advanced semiconductor production

這些詞彙與半導體、晶片製造及科技產業密切相關。

需要注意的是，Zero-shot Classification 的結果並不是人工標註答案，而是模型根據語意推論所得出的預測。  
因此，在正式應用中仍應搭配人工檢查、資料驗證或後續評估流程。